In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

In [3]:
iaa_labeled_b = pd.read_csv("/Users/marcoklaiber/Documents/06_Coding/Abschlussarbeiten/Konle MA/sentiment-analysis-football-scouting/data/labeled/iaa_subset_annotator_b_labeled.csv") 
print(len(iaa_labeled_b))
all_labeled_a = pd.read_csv("/Users/marcoklaiber/Documents/06_Coding/Abschlussarbeiten/Konle MA/sentiment-analysis-football-scouting/data/labeled/subset_balanced_750.csv")
print(len(all_labeled_a))

100
750


In [4]:
# Verwende nur die vom subset b
a_subset = all_labeled_a[all_labeled_a["comment_id"].isin(iaa_labeled_b["comment_id"])].copy()

In [5]:
print("Duplikate in A:", all_labeled_a["comment_id"].duplicated().sum())
print("Duplikate in B:", iaa_labeled_b["comment_id"].duplicated().sum())

Duplikate in A: 0
Duplikate in B: 0


In [6]:
# Merge der beiden subsets
iaa_merged = a_subset[["comment_id", "label"]].merge(
    iaa_labeled_b[["comment_id", "label"]],
    on="comment_id",
    how="inner",
    suffixes=("_a", "_b")
)

iaa_merged.head(5)

,comment_id,label_a,label_b
0,DE_0011,POSITIVE,NEGATIVE
1,DE_0019,POSITIVE,POSITIVE
2,DE_0041,POSITIVE,POSITIVE
3,DE_0043,POSITIVE,POSITIVE
4,DE_0063,POSITIVE,POSITIVE


In [7]:
kappa = cohen_kappa_score(
    iaa_merged["label_a"],
    iaa_merged["label_b"]
)

print("Cohen's Kappa:", kappa)

Cohen's Kappa: 0.7899159663865546


In [8]:
agreement = (iaa_merged["label_a"] == iaa_merged["label_b"]).mean()
print("Agreement:", round(agreement * 100, 2), "%")

Agreement: 86.0 %


In [9]:
labels = sorted(set(iaa_merged["label_a"]) | set(iaa_merged["label_b"]))

cm = confusion_matrix(
    iaa_merged["label_a"],
    iaa_merged["label_b"],
    labels=labels
)

cm_df = pd.DataFrame(cm, index=[f"A_{l}" for l in labels], columns=[f"B_{l}" for l in labels])
cm_df

,B_NEGATIVE,B_NEUTRAL,B_POSITIVE
A_NEGATIVE,26,4,2
A_NEUTRAL,0,30,2
A_POSITIVE,1,5,30


In [10]:
disagreements = iaa_merged[iaa_merged["label_a"] != iaa_merged["label_b"]].copy()
print("Unterschiedlich:", len(disagreements))
disagreements

Unterschiedlich: 14


,comment_id,label_a,label_b
0,DE_0011,POSITIVE,NEGATIVE
6,DE_0082,POSITIVE,NEUTRAL
18,DE_0235,NEGATIVE,NEUTRAL
31,EN_0061,POSITIVE,NEUTRAL
32,EN_0062,POSITIVE,NEUTRAL
43,EN_0136,NEGATIVE,POSITIVE
56,ES_0046,POSITIVE,NEUTRAL
58,ES_0055,POSITIVE,NEUTRAL
69,ES_0134,NEGATIVE,NEUTRAL
71,ES_0157,NEGATIVE,POSITIVE


In [16]:
comm_ids = disagreements["comment_id"].to_list()

df_disagreements = all_labeled_a[all_labeled_a["comment_id"].isin(comm_ids)].copy()
df_disagreements

,comment_id,language,club,player,comment,comment_de,label
10,DE_0011,DE,bayer04,Florian Wirtz,Boni and Wirtz are Disastrous 👏🙌,Boni and Wirtz are Disastrous 👏🙌,POSITIVE
81,DE_0082,DE,bayer04,Victor Boniface,If no be boniface I no go deal,If no be boniface I no go deal,POSITIVE
234,DE_0235,DE,fcbayern,Dayot Upamecano,"Instead of Kim, start him alongside Upamecano 😂","Instead of Kim, start him alongside Upamecano 😂",NEGATIVE
351,EN_0061,EN,liverpoolfc,Ibrahima Konate,"We talking about Konate, stop promoting Garbag...","Wir reden über Konate, hör auf, Garbagebriel z...",POSITIVE
352,EN_0062,EN,avfc,Ollie Watkins,O do watkins ficou braba,O do Watkins ist sauer geworden.,POSITIVE
426,EN_0136,EN,avfc,Youri Tielemans,Who voted for this? We were at the game. All o...,Wer hat dafür gestimmt? Wir waren beim Spiel. ...,NEGATIVE
531,ES_0046,ES,villarealcf,Raul Albiol,Raul Albiol sa deunouu ndeye heure no yewoo 😭😭...,Raul Albiol sa deunouu ndeye heure no yewoo 😭😭...,POSITIVE
540,ES_0055,ES,villarealcf,Pape Gueye,Como no ponéis a pape gueye q lo llevo en el F...,Warum setzt ihr Pape Gueye nicht ein? Ich habe...,POSITIVE
619,ES_0134,ES,villarealcf,Logan Costa,Logan Costa tu és um caso sério,"Logan Costa, du bist ein ernsthaftes Thema.",NEGATIVE
642,ES_0157,ES,sevillafc,Kelechi Iheanacho,Qué tengo que hacer para que no me toque la ca...,"Was muss ich tun, damit ich nicht das Trikot v...",NEGATIVE
